# 🌊 Samudrataṭa (समुद्रतट) — Coastal Digital Twin
## One-Click TA Evaluation & Inference Demo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flykrth/samudratata/blob/main/notebooks/main.ipynb)
[![Release: v1.0.0-Digital-Twin](https://img.shields.io/badge/Release-v1.0.0--Digital--Twin-blue.svg)](https://github.com/flykrth/samudratata/releases/tag/v1.0.0-Digital-Twin)
[![GitHub Repository](https://img.shields.io/badge/GitHub-flykrth%2Fsamudratata-green.svg)](https://github.com/flykrth/samudratata)

> **Course:** 22AIE304 — Deep Learning  
> **Project:** Multimodal Spatio-Temporal Digital Twin for Coastal Vulnerability Assessment  
> **Study Coastlines:** South India (Chellanam, Alappuzha, Nagapattinam, Cuddalore, Visakhapatnam)  

---

### 📌 Purpose of this Demo Notebook
Teaching Assistants and evaluators often do not have the time to install complex PyTorch Geometric environments locally to verify code. 
This standalone notebook provides a **100% self-contained, one-click execution environment** that:
1. Installs minimal dependencies (standard PyTorch + pure-Python PyG).
2. **Automatically downloads** the frozen dataset and pre-trained model weights from the official [v1.0.0-Digital-Twin Release](https://github.com/flykrth/samudratata/releases/tag/v1.0.0-Digital-Twin).
3. Executes a **single forward pass** of the **Spatio-Temporal GConvLSTM Digital Twin** on the test set to generate the **Geographic Coastal Vulnerability Map**.
4. Executes a **single forward pass** of the **Tidal Surge BiLSTM + Attention** network to extract multi-head temporal attention weights and generate the **Attention Heatmap**.


---
### 🛠️ Step 1: Install Minimal Dependencies
We only require standard `torch_geometric`, `h5py`, `matplotlib`, `shapely`, and `requests`. No C++ compilation or custom CUDA wheels needed!


In [ ]:
# Minimal dependency installation for Google Colab / local Jupyter
!pip install -q torch_geometric h5py matplotlib shapely requests tqdm
print("✓ Environment dependencies satisfied!")


---
### 📦 Step 2: Automated Asset Download from GitHub Releases
Downloads the frozen PyG graph dataset, ERA5/INCOIS ocean reanalysis time-series, pre-trained model checkpoints, and coastline boundary geometries directly from GitHub Release `v1.0.0-Digital-Twin`.


In [ ]:
import os
import urllib.request

RELEASE_URL = "https://github.com/flykrth/samudratata/releases/download/v1.0.0-Digital-Twin"
RAW_REPO_URL = "https://raw.githubusercontent.com/flykrth/samudratata/main"

ASSETS = {
    "data/south_india_coastal_graph.pt": f"{RELEASE_URL}/south_india_coastal_graph.pt",
    "data/ocean/ocean_timeseries_72h.h5": f"{RELEASE_URL}/ocean_timeseries_72h.h5",
    "weights/gconvlstm_best.pt": f"{RELEASE_URL}/gconvlstm_best.pt",
    "weights/surge_lstm_best.pt": f"{RELEASE_URL}/surge_lstm_best.pt",
    "data/coastline/south_india_coastline.geojson": f"{RAW_REPO_URL}/data/coastline/south_india_coastline.geojson",
}

print("Synchronizing required dataset & checkpoint assets from GitHub...")
for local_path, url in ASSETS.items():
    os.makedirs(os.path.dirname(local_path), exist_ok=True)
    if not os.path.exists(local_path) or os.path.getsize(local_path) == 0:
        print(f"Downloading {local_path}...")
        urllib.request.urlretrieve(url, local_path)
        size_mb = os.path.getsize(local_path) / (1024 * 1024)
        print(f"✓ Downloaded {local_path} ({size_mb:.2f} MB)")
    else:
        size_mb = os.path.getsize(local_path) / (1024 * 1024)
        print(f"✓ Cached: {local_path} ({size_mb:.2f} MB)")

print("
✓ All assets ready for immediate inference!")


---
### 🧠 Step 3: Self-Contained Neural Architectures
To prevent any environment mismatch or external dependency breakages, the exact model architectures (`CoastalGConvLSTM` and `SurgeBiLSTMAttention`) are defined directly below using standard PyTorch and PyG layers:
- **`GConvLSTM`**: Chebyshev polynomial spatio-temporal graph convolutional LSTM.
- **`SurgeBiLSTMAttention`**: Bidirectional LSTM with Multi-Head Self-Attention for tidal storm surge forecasting.


In [ ]:
import torch
import torch.nn as nn
from torch_geometric.nn import ChebConv
from typing import Tuple, Dict, Any

# 1. Spatio-Temporal Graph Convolutional LSTM Cell
class GConvLSTM(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, K: int = 3):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.K = K

        # Chebyshev Graph Convolutions for Gates
        self.conv_x_i = ChebConv(in_channels, out_channels, K=K, normalization="sym")
        self.conv_h_i = ChebConv(out_channels, out_channels, K=K, normalization="sym")
        self.w_c_i = nn.Parameter(torch.Tensor(1, out_channels))
        self.b_i = nn.Parameter(torch.Tensor(1, out_channels))

        self.conv_x_f = ChebConv(in_channels, out_channels, K=K, normalization="sym")
        self.conv_h_f = ChebConv(out_channels, out_channels, K=K, normalization="sym")
        self.w_c_f = nn.Parameter(torch.Tensor(1, out_channels))
        self.b_f = nn.Parameter(torch.Tensor(1, out_channels))

        self.conv_x_c = ChebConv(in_channels, out_channels, K=K, normalization="sym")
        self.conv_h_c = ChebConv(out_channels, out_channels, K=K, normalization="sym")
        self.b_c = nn.Parameter(torch.Tensor(1, out_channels))

        self.conv_x_o = ChebConv(in_channels, out_channels, K=K, normalization="sym")
        self.conv_h_o = ChebConv(out_channels, out_channels, K=K, normalization="sym")
        self.w_c_o = nn.Parameter(torch.Tensor(1, out_channels))
        self.b_o = nn.Parameter(torch.Tensor(1, out_channels))

    def forward(self, X, edge_index, edge_weight=None, H=None, C=None):
        if H is None:
            H = torch.zeros(X.shape[0], self.out_channels, device=X.device)
        if C is None:
            C = torch.zeros(X.shape[0], self.out_channels, device=X.device)

        I = torch.sigmoid(self.conv_x_i(X, edge_index, edge_weight) + self.conv_h_i(H, edge_index, edge_weight) + (self.w_c_i * C) + self.b_i)
        F = torch.sigmoid(self.conv_x_f(X, edge_index, edge_weight) + self.conv_h_f(H, edge_index, edge_weight) + (self.w_c_f * C) + self.b_f)
        C_tilde = torch.tanh(self.conv_x_c(X, edge_index, edge_weight) + self.conv_h_c(H, edge_index, edge_weight) + self.b_c)
        C = F * C + I * C_tilde
        O = torch.sigmoid(self.conv_x_o(X, edge_index, edge_weight) + self.conv_h_o(H, edge_index, edge_weight) + (self.w_c_o * C) + self.b_o)
        H = O * torch.tanh(C)
        return H, C


# 2. Coastal Digital Twin Model
class CoastalGConvLSTM(nn.Module):
    def __init__(self, in_channels: int = 192, hidden_channels: int = 64, K: int = 3, dropout: float = 0.15):
        super().__init__()
        self.gconv_lstm = GConvLSTM(in_channels=in_channels, out_channels=hidden_channels, K=K)
        self.norm = nn.LayerNorm(hidden_channels)
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Sequential(
            nn.Linear(hidden_channels, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid(),  # Bounds Vulnerability Score to [0.0, 1.0]
        )

    def forward(self, x_seq, edge_index, edge_weight=None):
        T, N, _ = x_seq.shape
        H, C = None, None
        for t in range(T):
            H, C = self.gconv_lstm(x_seq[t], edge_index, edge_weight, H=H, C=C)
        H = self.norm(H)
        H_drop = self.dropout(H)
        vulnerability_score = self.head(H_drop).squeeze(-1)
        return vulnerability_score, H


# 3. Oceanographic BiLSTM + Multi-Head Attention
class SurgeBiLSTMAttention(nn.Module):
    def __init__(self, input_dim: int = 7, hidden_dim: int = 64, num_layers: int = 2, num_heads: int = 4, dropout: float = 0.15):
        super().__init__()
        self.embed_dim = hidden_dim * 2
        self.bilstm = nn.LSTM(input_size=input_dim, hidden_size=hidden_dim, num_layers=num_layers, bidirectional=True, batch_first=True)
        self.norm1 = nn.LayerNorm(self.embed_dim)
        self.self_mha = nn.MultiheadAttention(embed_dim=self.embed_dim, num_heads=num_heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(self.embed_dim)
        self.dropout = nn.Dropout(dropout)
        self.query_proj = nn.Linear(self.embed_dim, self.embed_dim)
        self.pool_mha = nn.MultiheadAttention(embed_dim=self.embed_dim, num_heads=num_heads, dropout=dropout, batch_first=True)
        self.head = nn.Sequential(
            nn.Linear(self.embed_dim * 2, self.embed_dim),
            nn.LayerNorm(self.embed_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(self.embed_dim, self.embed_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout / 2.0),
            nn.Linear(self.embed_dim // 2, 1),
        )

    def forward(self, x):
        H, _ = self.bilstm(x)
        H_norm = self.norm1(H)
        H_attn, _ = self.self_mha(H_norm, H_norm, H_norm, need_weights=True, average_attn_weights=False)
        H_seq = self.norm2(H + self.dropout(H_attn))
        q = self.query_proj(H_seq[:, -1:, :])
        context, pool_weights = self.pool_mha(q, H_seq, H_seq, need_weights=True, average_attn_weights=False)
        head_weights = pool_weights.squeeze(2)
        consensus_weights = head_weights.mean(dim=1)
        context_vec = context.squeeze(1)
        final_state = H_seq[:, -1, :]
        fused = torch.cat([context_vec, final_state], dim=-1)
        y_pred = self.head(fused)
        return y_pred, {"weights": consensus_weights, "head_weights": head_weights}

print("✓ Architecture classes compiled successfully!")


---
### 🗺️ Step 4: Run Single Forward Pass — Coastal Digital Twin Vulnerability Map
Here we:
1. Load `south_india_coastal_graph.pt` (718 coastal transects, 2,134 spatial adjacency edges across 5 study zones).
2. Load pre-trained `weights/gconvlstm_best.pt`.
3. Execute a **single forward pass** across the multi-temporal graph.
4. Render the **publication-grade Coastal Vulnerability Map** across the 5 zones.


In [ ]:
import json
import time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Executing inference on device: {device}")

# 1. Load Graph and Weights
pyg_data = torch.load("data/south_india_coastal_graph.pt", map_location=device, weights_only=False)
ckpt = torch.load("weights/gconvlstm_best.pt", map_location=device, weights_only=False)

model = CoastalGConvLSTM(in_channels=192, hidden_channels=64, K=3, dropout=0.0).to(device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

# 2. Prepare Graph Input & Distance-Decay Edge Weights
dist_m = pyg_data.edge_attr[:, 0].to(device)
edge_weight = 1.0 / (torch.clamp(dist_m, min=50.0) / 100.0)
edge_weight = edge_weight / edge_weight.mean()

# Feature sequence: (T=6 years, N=718 nodes, D=192 latent features)
x_seq = pyg_data.x[:, 2:].unsqueeze(0).repeat(6, 1, 1).to(device)
edge_index = pyg_data.edge_index.to(device)

# 3. Single Forward Pass
t0 = time.time()
with torch.no_grad():
    vulnerability_scores, embeddings = model(x_seq, edge_index, edge_weight)
latency_ms = (time.time() - t0) * 1000.0

pred = vulnerability_scores.cpu().numpy()
pos = pyg_data.pos.cpu().numpy()
zone_names = pyg_data.zone_names
zone_idx = pyg_data.zone_idx.cpu().numpy()

print(f"✓ Single Forward Pass Complete in {latency_ms:.2f} ms!")
print(f"  Nodes Evaluated: {len(pred)}")
print(f"  Vulnerability Index: Mean = {pred.mean():.3f} | Min = {pred.min():.3f} | Max = {pred.max():.3f}")
print(f"  Critical Risk (>0.65): {(pred > 0.65).sum()} transects ({(pred > 0.65).mean()*100:.1f}%)")
print(f"  Moderate Risk (0.35-0.65): {((pred >= 0.35) & (pred <= 0.65)).sum()} transects")
print(f"  Low Risk (<0.35): {(pred < 0.35).sum()} transects")

# 4. Render Multi-Panel Geographic Vulnerability Map
fig = plt.figure(figsize=(18, 11), facecolor="#f8fafc")
gs = gridspec.GridSpec(2, 4, figure=fig, hspace=0.35, wspace=0.3)

# Panel A: Regional South India Coordinate Plane
ax_macro = fig.add_subplot(gs[:, :2])
ax_macro.set_facecolor("#f1f5f9")
ax_macro.plot([76.0, 75.0, 76.5, 77.5, 79.8, 80.3, 83.3, 85.0],
              [12.5, 14.0, 8.1, 8.0, 9.8, 13.0, 17.7, 19.5],
              color="#94a3b8", linestyle="--", lw=1.5, label="Coastline Baseline")

sc = ax_macro.scatter(pos[:, 0], pos[:, 1], c=pred, cmap="turbo", s=75, vmin=0.0, vmax=1.0,
                      edgecolor="#0f172a", linewidth=0.6, zorder=5)
ax_macro.set_title("A. South India Coastal Digital Twin Overview\nGConvLSTM Vulnerability Predictions (N=718)",
                   fontsize=12, fontweight="bold", pad=10)
ax_macro.set_xlabel("Longitude (°E)", fontsize=10, fontweight="bold")
ax_macro.set_ylabel("Latitude (°N)", fontsize=10, fontweight="bold")
ax_macro.grid(True, linestyle=":", alpha=0.6)
ax_macro.legend(loc="lower left", fontsize=9)

cbar = fig.colorbar(sc, ax=ax_macro, orientation="horizontal", pad=0.08, shrink=0.85)
cbar.set_label("Vulnerability Index (0.0: Resilient → 1.0: Critical Risk)", fontsize=10, fontweight="bold")

# Panels B-E: Zone Insets
display_zones = ["Chellanam", "Alappuzha", "Nagapattinam", "Visakhapatnam"]
for idx, z_name in enumerate(display_zones):
    z_i = zone_names.index(z_name)
    r, c = divmod(idx, 2)
    ax_z = fig.add_subplot(gs[r, 2 + c])
    ax_z.set_facecolor("#f8fafc")
    mask = zone_idx == z_i
    
    sc_z = ax_z.scatter(pos[mask, 0], pos[mask, 1], c=pred[mask], cmap="turbo", s=110,
                        vmin=0.0, vmax=1.0, edgecolor="#0f172a", linewidth=0.7, zorder=5)
    mean_v = pred[mask].mean()
    high_pct = (pred[mask] > 0.65).mean() * 100.0
    ax_z.set_title(f"{chr(66 + idx)}. {z_name}\nMean V={mean_v:.2f} | Critical: {high_pct:.0f}%",
                   fontsize=10, fontweight="bold", pad=8)
    ax_z.set_xlabel("Lon (°E)", fontsize=8)
    ax_z.set_ylabel("Lat (°N)", fontsize=8)
    ax_z.grid(True, linestyle=":", alpha=0.5)

plt.suptitle("Samudrataṭa: Spatio-Temporal GConvLSTM Coastal Vulnerability Inference", fontsize=15, fontweight="bold", y=0.98)
plt.show()


---
### 🌊 Step 5: Run Single Forward Pass — Tidal Surge Attention Heatmap
Here we:
1. Load `data/ocean/ocean_timeseries_72h.h5` (72-hour lookback window, 7 variables: SWH, wave period, wind components, pressure, SST).
2. Load pre-trained `weights/surge_lstm_best.pt`.
3. Execute a **single forward pass** on a real high-energy surge event (Chellanam monsoon surge).
4. Extract the 4-Head Attention weights and render the **Multi-Head Attention Heatmap**.


In [ ]:
import h5py

# 1. Load Model
surge_ckpt = torch.load("weights/surge_lstm_best.pt", map_location=device, weights_only=False)
surge_model = SurgeBiLSTMAttention(input_dim=7, hidden_dim=64, num_layers=2, num_heads=4, dropout=0.0).to(device)
surge_model.load_state_dict(surge_ckpt["model_state_dict"])
surge_model.eval()

# 2. Ingest Sample 72-Hour Oceanographic Window
with h5py.File("data/ocean/ocean_timeseries_72h.h5", "r") as hf:
    # Extract high-surge event from Chellanam zone
    x_norm = hf["Chellanam"]["X_zscore"][120:121]  # (1, 72, 7)
    x_raw = hf["Chellanam"]["X_raw"][120]          # (72, 7)
    timestamp = hf["Chellanam"]["timestamps_end"][120]
    if isinstance(timestamp, bytes):
        timestamp = timestamp.decode("utf-8")

x_tensor = torch.from_numpy(x_norm).float().to(device)

# 3. Single Forward Pass
t0 = time.time()
with torch.no_grad():
    pred_surge, attn_dict = surge_model(x_tensor)
latency_surge_ms = (time.time() - t0) * 1000.0

weights_heads = attn_dict["head_weights"].squeeze(0).cpu().numpy() # (4, 72)
consensus_weights = attn_dict["weights"].squeeze(0).cpu().numpy()   # (72,)
hours = np.arange(-71, 1)

print(f"✓ Surge BiLSTM Forward Pass Complete in {latency_surge_ms:.2f} ms!")
print(f"  Event Timestamp: {timestamp}")
print(f"  Predicted Surge Value (normalized): {pred_surge.item():.4f}")
print(f"  Attention Tensor: 4 heads across 72 antecedent hours")

# 4. Render Multi-Head Attention Heatmap
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), dpi=150, sharex=True,
                               gridspec_kw={"height_ratios": [1.0, 1.2]})

# Panel 1: Multi-Head Heatmap
im = ax1.imshow(weights_heads, aspect="auto", cmap="magma", extent=[-72, 0, 4.5, 0.5])
ax1.set_yticks([1, 2, 3, 4])
ax1.set_yticklabels(["Head 1\n(Swell Focus)", "Head 2\n(Wind Stress)", "Head 3\n(Pressure Drop)", "Head 4\n(Tidal Cycle)"], fontsize=9, fontweight="bold")
ax1.set_ylabel("Attention Head", fontsize=11, fontweight="bold")
ax1.set_title(f"A. Multi-Head Self-Attention Weights across 72h Lookback (Chellanam Event: {timestamp})",
              fontsize=12, fontweight="bold", pad=10)
cbar = fig.colorbar(im, ax=ax1, orientation="vertical", pad=0.02)
cbar.set_label("Attention Weight", fontsize=9)
ax1.grid(axis="x", color="#ffffff", linestyle=":", alpha=0.3)

# Panel 2: Consensus Attention vs Wave Height
ax2.plot(hours, consensus_weights, color="#dc2626", lw=2.5, label="Consensus Attention Weight")
ax2.set_ylabel("Attention Weight", color="#dc2626", fontsize=11, fontweight="bold")
ax2.tick_params(axis="y", labelcolor="#dc2626")
ax2.set_xlabel("Hours Prior to Surge Peak (t = 0)", fontsize=11, fontweight="bold")
ax2.grid(True, linestyle=":", alpha=0.6)

# Overlay Significant Wave Height (SWH) on twin axis
ax2_twin = ax2.twinx()
swh = x_raw[:, 0]
ax2_twin.plot(hours, swh, color="#2563eb", lw=2.0, linestyle="--", label="Significant Wave Height (m)")
ax2_twin.set_ylabel("Significant Wave Height (m)", color="#2563eb", fontsize=11, fontweight="bold")
ax2_twin.tick_params(axis="y", labelcolor="#2563eb")

# Combined Legend
lines_1, labels_1 = ax2.get_legend_handles_labels()
lines_2, labels_2 = ax2_twin.get_legend_handles_labels()
ax2.legend(lines_1 + lines_2, labels_1 + labels_2, loc="upper left", frameon=True, fontsize=10)
ax2.set_title("B. Consensus Attention Alignment with Wave Energy Accumulation", fontsize=11, fontweight="bold", pad=8)

plt.tight_layout()
plt.show()


---
### 📋 Step 6: Architecture Verification & Metrics Summary

| Component | Model Architecture | Inputs | Test Metric | Inference Latency |
| :--- | :--- | :--- | :--- | :--- |
| **Spatial-Temporal Digital Twin** | `CoastalGConvLSTM` ($K=3$, $\text{dim}=64$) | 718 transects, 2,134 edges, 192 latents | $R^2 = 0.719$, $\text{RMSE}=0.117$ | **< 20 ms** |
| **Ocean Storm Surge** | `SurgeBiLSTMAttention` (2 layers, 4 heads) | 72h window $\times$ 7 met-ocean features | $\text{MAE}=0.071$, $\rho=0.88$ | **< 10 ms** |
| **Visual Change Detection** | `ChangeFormerV6` (Pretrained Siamese ViT) | Bitemporal Sentinel-2 (256x256) | $\text{F1}=0.785$, $\text{IoU}=0.648$ | — |

### Key Conclusions:
- **Zero-Friction Reproducibility:** Single forward passes execute out-of-the-box on both CPU and GPU.
- **Physical Fidelity:** The Spatio-Temporal GConvLSTM accurately isolates high-vulnerability hotspots (Chellanam & Nagapattinam) while recognizing lower-risk armored zones.
- **Explainable Attention:** The multi-head attention mechanism demonstrates peak activation 12 to 24 hours prior to surge peaks, aligning with oceanographic swell propagation physics.
